In [1]:
!pip install pyarrow

   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/27.6 MB ? eta -:--:--
   - -------------------------------------- 1.3/27.6 MB 3.9 MB/s eta 0:00:07
   --- ------------------------------------ 2.6/27.6 MB 4.9 MB/s eta 0:00:06
   ----- ---------------------------------- 3.7/27.6 MB 5.2 MB/s eta 0:00:05
   ------- -------------------------------- 5.5/27.6 MB 5.9 MB/s eta 0:00:04
   ----------- ---------------------------- 7.9/27.6 MB 6.9 MB/s eta 0:00:03
   --------------- ------------------------ 10.7/27.6 MB 8.0 MB/s eta 0:00:03
   ------------------ --------------------- 13.1/27.6 MB 8.4 MB/s eta 0:00:02
   ------------------------ --------------- 16.8/27.6 MB 9.4 MB/s eta 0:00:02
   ------------------------------- -------- 21.8/27.6 MB 11.0 MB/s eta 0:00:01
   ------------------------------------- -- 26.0/27.6 MB 12.0 MB/s eta 0:00:01
   ---------------------------------------- 27.6/27.6 MB 12.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# --- 1. LIBRARIES ---
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_curve, auc)

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Dense, Conv1D, BatchNormalization, 
                                     concatenate, Flatten, Dropout, Attention, 
                                     Reshape, GlobalAveragePooling1D, Multiply, MaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
# --- 2. REPRODUCIBILITY (The "Same Result" Rule) ---
def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    print(f"Environment seeds locked at {seed}")

set_seeds(42) 

Environment seeds locked at 42


In [4]:
# Load the specific binary dataset
df = pd.read_csv("leave_Caida2007.csv", engine="pyarrow")

# Verify the load by checking the total rows and columns
print(f"Dataset successfully loaded.")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

Dataset successfully loaded.
Total Rows: 100668668
Total Columns: 84


In [5]:
df.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
       'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
       'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std',
       'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean',
       'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot',
       'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
       'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',
       'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags',
       'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Packets/s',
       'Bwd Packets/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean',
       'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt',
       'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt',
       'CWE Flag Count

In [6]:
# 1. Strip any leading/trailing whitespace and convert to lowercase
df.columns = df.columns.str.strip().str.lower()

# 2. Replace spaces, slashes, and dots with underscores for easy coding
df.columns = df.columns.str.replace(' ', '_', regex=False)
df.columns = df.columns.str.replace('/', '_', regex=False)
df.columns = df.columns.str.replace('.', '_', regex=False)

# 3. Print the first 10 columns to verify the change
print("First 10 standardized columns:")
print(df.columns[:10].tolist())

# 4. Find the exact name of your Label column
label_col = [col for col in df.columns if 'label' in col]
print(f"\nLabel column found: {label_col}")

First 10 standardized columns:
['flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts']

Label column found: ['label']


In [7]:
# List of columns that are not useful for training features
identifiers = ['flow_id', 'src_ip', 'dst_ip', 'timestamp']

# Drop them only if they exist in the current dataframe
df.drop(columns=[col for col in identifiers if col in df.columns], inplace=True)

print(f"Identifiers removed.")
print(f"Remaining columns: {df.shape[1]}")

Identifiers removed.
Remaining columns: 80


In [8]:
rows_with_nan = df.isna().any(axis=1).sum()

print(f"Total rows with at least one NaN: {rows_with_nan}")

Total rows with at least one NaN: 1403554


In [9]:
# ==========================================================
import joblib
import gc
import numpy as np

print("Loading Pre-trained Regression Imputer...")

# 1. Replace all Inf with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 2. Load the saved models and column lists
imputer_data = joblib.load("regression_imputer.pkl")
imputer_models = imputer_data['models']
COLS_WONA = imputer_data['cols_wona']
COLS_WNA = imputer_data['cols_wna']

chunk_size = 5_000_000

# 3. Predict and Fill NaNs in chunks (No training, just inference)
for target_col in COLS_WNA:
    if target_col in imputer_models:
        print(f"Applying predictions for: {target_col}")
        model = imputer_models[target_col]
        predict_idx = df.index[df[target_col].isna()]
        
        for i in range(0, len(predict_idx), chunk_size):
            idx_chunk = predict_idx[i : i + chunk_size]
            X_pred_chunk = df.loc[idx_chunk, COLS_WONA].values
            
            # Predict and assign
            preds = model.predict(X_pred_chunk)
            df.loc[idx_chunk, target_col] = preds
            
            del X_pred_chunk, preds
            
    # Force memory cleanup after each column
    gc.collect()

print(f"Imputation Complete. Total NaN remaining: {df.isna().sum().sum()}")
# ==========================================================

Loading Pre-trained Regression Imputer...
Applying predictions for: flow_byts_s


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Applying predictions for: flow_pkts_s


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Imputation Complete. Total NaN remaining: 0


In [10]:
# Check how many duplicates exist
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

# Drop duplicates if any
df = df.drop_duplicates().reset_index(drop=True)
print(f"Data shape after dropping duplicates: {df.shape}")


Number of duplicate rows: 2300611
Data shape after dropping duplicates: (98368057, 80)


In [11]:
# 1. Check for any columns that are not integers or floats (excluding 'label')
non_numeric = df.drop(columns=['label']).select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns found (excluding label): {non_numeric}")

# 2. Check for Infinite values (common in this dataset)
inf_count = np.isinf(df.select_dtypes(include=[np.number])).values.sum()
print(f"Total Infinite values: {inf_count}")

# 3. Check for Null/NaN values
nan_count = df.isnull().sum().sum()
print(f"Total NaN values: {nan_count}")

Non-numeric columns found (excluding label): []
Total Infinite values: 0
Total NaN values: 0


In [12]:
# 1. Check unique values in the label column
print("Unique labels in dataset:")
print(df['label'].unique())

# 2. Check the count of each label to see if it is imbalanced
print("\nLabel counts:")
print(df['label'].value_counts())

# 3. Check the data type of the label
print(f"\nLabel data type: {df['label'].dtype}")

Unique labels in dataset:
[1 0]

Label counts:
label
1    88773177
0     9594880
Name: count, dtype: int64

Label data type: int64


In [13]:
import joblib
import gc
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

print("Starting Splitting and Scaling Phase...")

# 1. Define Features (X) and Target (y)
X = df.drop(columns=['label'])
y = df['label']

# CLEAR RAM: Delete original dataframe
del df
gc.collect()

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# CLEAR RAM: Delete X
del X
gc.collect()

print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set:  {X_test.shape[0]} rows")

# 3. Initialize the Scaler and transform training data
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

# CLEAR RAM: Delete unscaled X_train immediately
del X_train
gc.collect()

# 4. Transform testing data
X_test_scaled = scaler.transform(X_test)

# CLEAR RAM: Delete unscaled X_test immediately
del X_test
gc.collect()

# 5. SAVE SCALER IMMEDIATELY
joblib.dump(scaler, "scaler.pkl")
print("Scaling Complete and scaler.pkl saved successfully!")

Starting Splitting and Scaling Phase...
Training set: 78694445 rows
Testing set:  19673612 rows
Scaling Complete and scaler.pkl saved successfully!


In [14]:
# --- STAGE 1: AUTOENCODER DEFINITION ---

# 1. Use your 79 features as the input dimension
input_dim = X_train_scaled.shape[1]
ae_in = Input(shape=(input_dim,))

# 2. Encoder: Bottleneck strategy (96 neurons -> 32 neurons)
enc = Dense(96, activation='relu')(ae_in)
enc = BatchNormalization()(enc)
enc = Dense(32, activation='relu')(enc) 

# 3. AE Attention Mechanism
# Reshaping to (1, 32) allows the Attention layer to weigh the encoded features
att_reshape = Reshape((1, 32))(enc)
att_logic = Attention()([att_reshape, att_reshape])
att_flat = Flatten()(att_logic)

# 4. Decoder: Reconstruction strategy
# Maps the 32 features back to the original 79 dimensions
dec = Dense(96, activation='tanh')(att_flat)
dec_out = Dense(input_dim, activation='sigmoid')(dec) 

# 5. Model Creation
# 'autoencoder' is for training; 'encoder_only' is for feature extraction
autoencoder = Model(ae_in, dec_out, name="Autoencoder")
encoder_only = Model(ae_in, att_flat, name="Encoder_Extractor")

# 6. Compilation
# Using Adam optimizer and Mean Absolute Error (MAE) loss
autoencoder.compile(optimizer='adam', loss='mae')

# Display the architecture
autoencoder.summary()

Model: "Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 79)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 96)                │           7,680 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 96)                │             384 │ dense[0][0]                │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 32)                │           3,104 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ reshape (Reshape)             │ (None, 1, 32)             │               0 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention (Attention)         │ (None, 1, 32)             │               0 │ reshape[0][0],             │
│                               │                           │                 │ reshape[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 32)                │               0 │ attention[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 96)                │           3,168 │ flatten[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 79)                │           7,663 │ dense_2[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 21,999 (85.93 KB)

 Trainable params: 21,807 (85.18 KB)

 Non-trainable params: 192 (768.00 B)

In [15]:
# 1. Manually split validation data so Keras doesn't duplicate RAM
from sklearn.model_selection import train_test_split

X_train_final, X_val_final = train_test_split(
    X_train_scaled, test_size=0.2, random_state=42
)

# CLEAR RAM: Delete the pre-split array
del X_train_scaled
gc.collect()

# 2. Define Early Stopping
early_stop_ae = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# 3. Train the model using explicit validation_data
print("Starting Autoencoder Training...")
history_ae = autoencoder.fit(
    X_train_final, X_train_final, # Input is also target
    epochs=30,
    batch_size=1024,
    validation_data=(X_val_final, X_val_final), # Replaces validation_split
    callbacks=[early_stop_ae],
    verbose=1
)

print("\nAutoencoder training finished.")

Starting Autoencoder Training...
Epoch 1/30


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


61481/61481 ━━━━━━━━━━━━━━━━━━━━ 430s 6ms/step - loss: 0.0031 - val_loss: 0.0019
Epoch 2/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 283s 5ms/step - loss: 0.0018 - val_loss: 0.0018
Epoch 3/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 278s 4ms/step - loss: 0.0017 - val_loss: 0.0017
Epoch 4/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 278s 4ms/step - loss: 0.0015 - val_loss: 0.0015
Epoch 5/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 279s 4ms/step - loss: 0.0014 - val_loss: 0.0014
Epoch 6/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 286s 5ms/step - loss: 0.0014 - val_loss: 0.0014
Epoch 7/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 282s 5ms/step - loss: 0.0014 - val_loss: 0.0014
Epoch 8/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 277s 4ms/step - loss: 0.0014 - val_loss: 0.0014
Epoch 9/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 278s 4ms/step - loss: 0.0013 - val_loss: 0.0014
Epoch 10/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 277s 4ms/step - loss: 0.0013 - val_loss: 0.0014
Epoch 11/30
61481/61481 ━━━━━━━━━━━━━━━━━━━━ 277s 4ms/step - loss: 0.0013 - val_loss: 0.00

In [16]:
# 1. Save the Encoder (the 32-feature extractor)
encoder_only.save("AE_Encoder_Extractor.keras")

# 2. Save the full Autoencoder
autoencoder.save("AE_Stage1_Full.keras")

print("Autoencoder components saved successfully:")
print("- AE_Encoder_Extractor.keras")
print("- AE_Stage1_Full.keras")

Autoencoder components saved successfully:
- AE_Encoder_Extractor.keras
- AE_Stage1_Full.keras
